In [1]:
%env PYTHONPATH=/opt/micromamba/envs/sirius/repos/dev-packages:${PYTHONPATH}
import sys
sys.path.insert(0, "/opt/micromamba/envs/sirius/repos/dev-packages/siriuspy/")
sys.path.insert(1, "/home/ABTLUS/arnaldo.filho/repos/cax-scripts")

env: PYTHONPATH=/opt/micromamba/envs/sirius/repos/dev-packages:${PYTHONPATH}


In [2]:
from siriuspy.devices import CAXCtrl, CAXMirror
import matplotlib.pyplot as plt
import numpy as np

### Functions to calculate centroid, FWHM and average intensity at center.

In [316]:
THRESHOLD_ROI = 10
THRESHOLD = 1000

def beam_centroid(img, droi=2, threshold=THRESHOLD_ROI):
    """Calculate the beam centroid coordinates.
    
    Checks whether the average intensity in a ROI around the centroid is above a given threshold.
    """
    # Centroids.
    cx = np.sum(img, axis=0).argmax()
    cy = np.sum(img, axis=1).argmax()

    roi_peak = np.mean(img[cy - droi : cy + droi + 1,
                           cx - droi : cx + droi + 1])
    mean = np.mean(img)
    ratio_rtom = roi_peak / mean if mean != 0 else 0

    if ratio_rtom < threshold:
        return [None, None]

    return [cx, cy]


def beam_fwhm(img, cx, cy):
    """Calculate the FWHM in x and y directions."""
    # Get profiles.
    x_profile = img[cy, :]
    y_profile = img[:, cx]

    fwhm_x = np.sum(x_profile > (x_profile.max() / 2))
    fwhm_y = np.sum(y_profile > (y_profile.max() / 2))

    return fwhm_x, fwhm_y


def avg_intensity_center(cax):
    """Calcuate the average intensity in a ROI at the beam centroid.
    
    The intensity is normalized by the exposure time and the area of the ROI.

    Args:
        cax: CAXCtrl instance.

    Returns:
        img       : the image data.
        intensity : the ROI average intensity normalized by exposure time and area.
        cxy       : the coordinates of the beam centroid.
        fwhm      : the FWHM in x and y directions.
    """
    # Get image data and calculate centroid.
    img     = cax.dvf_B1.image
    exptime = cax.dvf_B1.exposure_time

    cxy = beam_centroid(img)
    if cxy == [None, None]:
        return (img, 0, 0, 0, [None, None], [None, None])
    cx, cy = cxy

    # ROI defined by a square with 2 sigmas as its side.
    fwhm_x, fwhm_y = beam_fwhm(img, cx, cy)

    # Masked image.
    # Consider only pixels above half of the peak value.
    dq = 2
    peak = np.mean(img[cy - dq : cy + dq + 1,
                       cx - dq : cx + dq + 1])
    
    mask = img > (peak / 2)
    area_mask  = np.sum(mask)
    img_masked = np.where(mask, img, 0)
    area_img_masked = np.sum(img_masked)

    mask_intensity = (area_img_masked / (area_mask * exptime)
                      if area_mask != 0 else 0)

    peak /= exptime
    peak_fwhm_norm = (peak / (fwhm_x * fwhm_y)
                      if fwhm_x * fwhm_y != 0 else 0)

    return (img, peak, peak_fwhm_norm,
            mask_intensity, cxy, [fwhm_x, fwhm_y])


def status_show(cax):
    """Show the beam status."""
    pixel_size = 0.48   # micrometers
    img, peak, peak_fwhm_norm, intensity, cxy, fwhm = avg_intensity_center(cax)

    print("FWHM                      :"
          f" (x : {fwhm[0]:.2f}, y : {fwhm[1]:.2f}) pixels"
          f" = (x : {fwhm[0] * pixel_size:.2f} µm, y : {fwhm[1] * pixel_size:.2f} µm)")
    print("Area FWHM                 :"
          f" {fwhm[0] * fwhm[1] * pixel_size ** 2:.4f} µm²")
    print("Beam centroid             :"
          f" (x = {cxy[0]:.2f}, y = {cxy[1]:.2f}) pixel")
    print("Peak value (t norm)       :"
          f" {peak:.2f}")
    print("Peak value (FWHM, t norm) :"
          f" {peak_fwhm_norm:.4f}")
    print("Average intensity         :"
          f" {intensity:.4e}")
    print(" (within 2 sigmas at beam centroid, normalized by exposure time and area)")


#### Beam walk procedure function.

In [17]:
import random
import time


def stop_all_motors(cax, devnames, delay=2):
    """Stop all specified devices."""
    for dev in devnames:
        dev_stop = f"{dev}_stop"
        setattr(cax.mirror, dev_stop, 1)
        time.sleep(0.5)

        is_moving = getattr(cax.mirror, f"{dev}_mvn")
        while is_moving:
            print(f"Waiting for {dev} to stop...")
            time.sleep(0.5)
            is_moving = getattr(cax.mirror, f"{dev}_mvn")

    time.sleep(delay)
    print(" ø All motors stopped.\n")

        
def beam_walk(cax, devnames, dsteps, nsteps, tolerance=0.01, delay=2):
    """Perform a beam walk by moving the CAX mirror.
      
    Perform measurements of the average intensity at the
    beam centroid to accept or reject new positions.
    """
    cm = cax.mirror
    lowlim = 1 - tolerance

    (img, peak_0, peak_fwhm_norm_0,
     intensity0, cxy, fwhm) = avg_intensity_center(cax)

    for ii in range(nsteps):
        # Choose device and direction.
        ndev  = len(devnames)
        rdev  = random.randint(0, ndev - 1)
        rsign = random.choice([-1, 1])
        motor = devnames[rdev]
        dstep = rsign * dsteps[motor]
        devstatus0 = getattr(cm, motor)

        # Guarantee all motors are stopped.
        stop_all_motors(cax, devnames, delay)

        print(f"\n\n *** Pass {ii + 1}, chosen device: {motor}")

        # Check current intensity.
        (_, peak_before, peak_fwhm_norm_before,
         intensity_before, cxy, fwhm) = avg_intensity_center(cax)

        # Calculate new position.
        newpos = devstatus0 + dstep
        print(f"\t current position = {devstatus0:.4e} "
              f" → newpos = {newpos:.4e}\n")

        # Attempt move; _PVAccessor validates against live hardware limits.
        try:
            setattr(cm, motor, newpos)
            time.sleep(delay)
        except ValueError as err:
            print(f"\n ERROR: {err} → new position REJECTED.\n Next step.\n")
            continue

        # New value is valid, change position.
        (img, peak_after, peak_fwhm_norm_after,
         intensity_after, cxy, fwhm) = avg_intensity_center(cax)

        # Check whether image was lost.
        if np.sum(img) < THRESHOLD:
            print("  > Image lost, change REJECTED."
                  " Back to former position.")
            setattr(cm, motor, devstatus0)
            time.sleep(delay)
            continue

        # Reject change if the variation in intensity is negative.
        # if (intensity_after < intensity_before * lowlim or
        #     intensity_after < intensity0 * lowlim):
        if (peak_after < peak_before * lowlim or
            peak_after < peak_0 * lowlim):
            print("\n  >> Intensity dropped, change REJECTED."
                  " Back to former position.")
            setattr(cm, motor, devstatus0)
            time.sleep(delay)
            continue

        # Position accepted.
        print(f" >> New position ACCEPTED within {tolerance * 100}%:\n"
              " \t   Peak intensity change ="
              f" {peak_after - peak_before:.4f}\n")

    # Guarantee all motors are stopped.
    stop_all_motors(cax, devnames, delay)

    (_, peak_end, peak_fwhm_norm_end,
     intensity_end, cxy, fwhm) = avg_intensity_center(cax)
    return peak_end - peak_0


In [15]:
cax = CAXCtrl()

In [44]:
devnames = ['tx', 'ry', 'y1', 'y2', 'y3']
stop_all_motors(cax, devnames)

 ø All motors stopped.



#### Initial status.

In [330]:
print("####\n#### Initial conditions:\n####\n")
status_show(cax)

####
#### Initial conditions:
####

FWHM                      : (x : 151.00, y : 80.00) pixels = (x : 72.48 µm, y : 38.40 µm)
Area FWHM                 : 2783.2320 µm²
Beam centroid             : (x = 1397.00, y = 1354.00) pixel
Peak value (t norm)       : 60960.00
Peak value (FWHM, t norm) : 5.0464
Average intensity         : 4.4454e+04
 (within 2 sigmas at beam centroid, normalized by exposure time and area)


### Main function.

In [ ]:
# dsteps = [dtx, dry, dy1, dy2, dy3]
# devnames = ['tx', 'ry', 'y1', 'y2', 'y3']
devnames = ['tx', 'ry']

# Steps for each device.
dtx = 0.003
dry = 0.003
dy1 = 0.001   # Leveler Z-
dy2 = 0.001   # Leveler X+
dy3 = 0.001   # Leveler Z+
dsteps = {'tx': dtx, 'ry': dry, 'y1': dy1, 'y2': dy2, 'y3': dy3}

# Parameters.
wait      = 5          # delay (s)
nsteps    = 10         # number of attempts
tolerance = 0.01       # tolerance (x 100%)

# Perform beam walk.
intensity_change = beam_walk(cax, devnames, dsteps, nsteps, tolerance, wait)
print("\n#####\n##### Intensity variation in the process ="
      f" {intensity_change}\n#####\n\n##### DONE.")

# Final state.
print("\n\n####\n##### Final conditions:\n####\n")
status_show(cax)

#### Final status.

In [325]:
print("####\n#### Final conditions:\n####\n")
status_show(cax)

####
#### Final conditions:
####

FWHM                      : (x : 178.00, y : 85.00) pixels = (x : 85.44 µm, y : 40.80 µm)
Area FWHM                 : 3485.9520 µm²
Beam centroid             : (x = 1635.00, y = 1362.00) pixel
Peak value (t norm)       : 217664.00
Peak value (FWHM, t norm) : 14.3863
Average intensity         : 1.5750e+05
 (within 2 sigmas at beam centroid, normalized by exposure time and area)


In [ ]:
# TEST for average intensity estimate.

# img, (cx, cy) = beam_centroid(cax)
# fwhm_x, fwhm_y = fwhm(img, cx, cy)
# exptime = cax.dvf_B1.exposure_time
img, caintens = avg_intensity_center(cax)

# print(f"Centroid: ({cx}, {cy})")
# print(f"FWHM: (x: {fwhm_x}, y: {fwhm_y})")
# print(f"Exposure Time: {exptime:.2f} s")
print(f"Center Average Intensity: {caintens:10.4f}")

plt.imshow(img)
plt.show()


In [ ]:
    # DEBUG
    # print(f"Sum of masked image  : {area_img_masked:.2f}")
    # print(f"Area                 : {area_mask:.2f} pixels")
    # mean_masked = area_img_masked / area_mask if area_mask != 0 else 0
    # print(f"Mean of masked image : {mean_masked:.2f}")
    # print(f"Exposure time        : {exptime:.2f} s")
    # print(f"Time normalized mean : {mean_masked / exptime:.2f}")
    # print(f"Intensity            : {intensity:.2f} counts/s/pixel")

    # print(f"\n Numpy mean of masked image : {np.mean(img_masked):.2f}")

    # fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # ax1.imshow(mask)
    # ax1.scatter(cx, cy, color='red', marker='x')
    # ax1.set_xlim(cx - 100, cx + 100)
    # ax1.set_ylim(cy + 100, cy - 100)
    # ax1.set_title(f"Centroid: ({cx:.2f}, {cy:.2f}), FWHM: (x: {fwhm_x:.2f}, y: {fwhm_y:.2f})")
    # fig.colorbar(ax1.images[0], ax=ax1, label='Intensity')

    # ax2.imshow(img_masked, cmap='viridis')
    # ax2.set_xlim(cx - 100, cx + 100)
    # ax2.set_ylim(cy + 100, cy - 100)
    # ax2.scatter(cx, cy, color='red', marker='x')
    # ax2.set_title(f"Centroid: ({cx:.2f}, {cy:.2f}), FWHM: (x: {fwhm_x:.2f}, y: {fwhm_y:.2f})")
    # fig.colorbar(ax2.images[0], ax=ax2, label='Intensity')

    # plt.show()
    # DEBUG
    # print(f"Area: {area_mask:.2f} pixels²")
    # print(f"Sum of masked image: {area_img_masked:.2f}")

    # Extract ROI and calculate average intensity.
    # img_at_roi = img[cy - droi_y : cy + droi_y + 1,
    #                  cx - droi_x : cx + droi_x + 1]
